# Lab 1 — DVM Color Classification

Классификация цвета автомобиля по фронтальному фото (датасет DVM, top-6 цветов: `black, grey, white, blue, silver, red`).

**Цель:** обучить и сравнить 3 модели (1 from-scratch + 2 pretrained), получить F1_macro > 0.8.

**Запуск:** `Runtime → Change runtime type → GPU`, затем выполни ячейки по порядку. Все ячейки идемпотентны — переисполнение не ломает состояние.

## 1. Dev — git pull/clone (опционально)

Обновляет `.py` файлы лабы из GitHub. Требует Colab-secret `GITHUB_TOKEN`; без токена ячейка молча скипнется.


In [ ]:
import os

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None

if TOKEN:
    REPO_DIR = "/content/ITMO-CV"
    if os.path.exists(REPO_DIR):
        !cd {REPO_DIR} && git pull
    else:
        !git clone https://{TOKEN}@github.com/Ma-XD/ITMO-CV.git {REPO_DIR}
    print(f"📂 dev clone/pull → {REPO_DIR}")
else:
    print("ℹ️  Нет GITHUB_TOKEN — пропускаю clone.")


## 2. Path setup

Ноутбук рассчитан на запуск из папки `lab1-CLAS/` (либо клон репо, либо распакованная папка). Делаем `cwd → lab1-CLAS` и добавляем её в `sys.path`, чтобы `from config import …` работал.


In [ ]:
import os
import sys

# В Colab папка может быть либо клоном (/content/ITMO-CV/lab1-CLAS/), либо
# распакована в /content/lab1-CLAS/. Локально — папка лабы.
CANDIDATES = [
    "/content/ITMO-CV/lab1-CLAS",
    "/content/lab1-CLAS",
    os.getcwd(),
]
for d in CANDIDATES:
    if os.path.exists(os.path.join(d, "config.py")):
        os.chdir(d)
        break
else:
    raise RuntimeError(
        "Не нашли config.py. Положи папку lab1-CLAS/ в /content/ "
        "(File panel в Colab) и перезапусти ячейку."
    )

sys.path.insert(0, os.getcwd())
print(f"📂 cwd: {os.getcwd()}")


## 3. Drive mount

Нужен только если хочешь сохранять `outputs/` (чекпоинты, графики) на Drive — иначе они остаются на локальном SSD VM и пропадут с сессией. Можно пропустить, если запускаешь без Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Зависимости

Ставим всё, что нужно лабе. `requirements.txt` лежит рядом с ноутбуком.


In [ ]:
!pip install -q -r requirements.txt


## 5. Verify окружения

In [ ]:
from env_config import print_env
print_env()

## 6. Загрузка датасета

Драйв медленный для случайного чтения мелких файлов (61k jpg). Поэтому архив **копируем** один раз с Drive на локальный SSD VM (`/content/`) и распаковываем туда же — обучение читает из `/content/data/dvm/...`, а на Drive ничего не пишем.

Источник: `MyDrive/ITMO-CV/lab1-CLAS/data/dvm_confirmed_fronts.zip` (765 МБ, без сжатия).

In [ ]:
%%bash
set -e
DATA_DIR=/content/data

# Ищем zip с датасетом в нескольких местах:
#   1) /content/drive/MyDrive/ITMO-CV/lab1-CLAS/data/  (dev, на Drive)
#   2) /content/dvm_confirmed_fronts.zip               (uploaded в Colab Files)
#   3) ./dvm_confirmed_fronts.zip                      (рядом с ноутбуком)
SRC=""
for cand in \
  /content/drive/MyDrive/ITMO-CV/lab1-CLAS/data/dvm_confirmed_fronts.zip \
  /content/dvm_confirmed_fronts.zip \
  ./dvm_confirmed_fronts.zip; do
  if [ -f "$cand" ]; then SRC="$cand"; break; fi
done

if [ -z "$SRC" ]; then
  echo "❌ Не нашёл dvm_confirmed_fronts.zip ни в одном из ожидаемых мест."
  echo "   Загрузи zip в /content/dvm_confirmed_fronts.zip (Colab → Files) и перезапусти."
  exit 1
fi
echo "📦 SRC: $SRC ($(numfmt --to=iec $(stat -c %s "$SRC")))"

DST=/content/dvm_confirmed_fronts.zip
src_size=$(stat -c %s "$SRC")
need_copy=true
if [ -f "$DST" ] && [ "$DST" != "$SRC" ]; then
  dst_size=$(stat -c %s "$DST")
  if [ "$src_size" = "$dst_size" ]; then
    echo "✅ Кэш уже есть: $DST"
    need_copy=false
  else
    echo "⚠️  Размеры не совпали ($dst_size ≠ $src_size) — перекопирую"
    rm -f "$DST"
    rm -rf "$DATA_DIR/dvm/confirmed_fronts"
  fi
elif [ "$DST" = "$SRC" ]; then
  need_copy=false
fi

if [ "$need_copy" = "true" ]; then
  echo "📥 cp → $DST"
  cp "$SRC" "$DST"
fi

mkdir -p "$DATA_DIR"
if [ ! -d "$DATA_DIR/dvm/confirmed_fronts" ]; then
  echo "📦 unzip → $DATA_DIR ..."
  unzip -q "$DST" -d "$DATA_DIR"
else
  echo "✅ Уже распаковано: $DATA_DIR/dvm/confirmed_fronts"
fi

echo "---"
echo "Изображений: $(find "$DATA_DIR/dvm/confirmed_fronts" -name '*.jpg' | wc -l)  (ожидаем 61827)"


## 7. Сборка `index.csv`

Парсит filename'ы по `$$`, фильтрует `unlisted`/`multicolour` и off-target цвета, оставляет только top-6 → `/content/data/dvm/index.csv`. Идемпотентно: если CSV уже есть, скрипт его не перезаписывает (для пересборки — `--force`).

In [ ]:
!python build_index.py


## 8. Характеристики моделей

Сводка по трём моделям: общее число параметров и (для pretrained) разбивка на head/backbone — это то, что пойдёт в две param-group'ы оптимизатора с разными LR в разделах 10–11.


In [ ]:
from models import build_model, get_param_groups
from config import ALL_MODELS, PRETRAINED_MODELS

for name in ALL_MODELS:
    model = build_model(name)
    n_total = sum(p.numel() for p in model.parameters())
    line = f"[{name:<20}] params={n_total/1e6:>5.2f}M"
    if name in PRETRAINED_MODELS:
        groups = get_param_groups(model, name)
        n_head = sum(p.numel() for p in groups[0]["params"])
        n_backbone = sum(p.numel() for p in groups[1]["params"])
        line += f"  |  head={n_head/1e3:>5.1f}K   backbone={n_backbone/1e6:>5.2f}M"
    print(line)
    del model


## 9. Train: custom_resnet18 (from-scratch)

ResNet-18, написанный руками. Обучается с нуля и одной param-group (head + backbone вместе, `LR=1e-3`). Scheduler — `CosineAnnealingLR`. 12 эпох — столько же, сколько fine-tune, для честного сравнения по budget'у.

Веса сохранятся в `CHECKPOINT_DIR/custom_resnet18_last.pth` и подхватятся в разделе 12.


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import MODEL_CUSTOM
from data import get_dataloaders, load_index, make_splits
from engine import fit
from env_config import get_device
from models import build_model, get_param_groups

EPOCHS = 12
LR = 1e-3
WEIGHT_DECAY = 1e-4
print(f"[{MODEL_CUSTOM}] epochs={EPOCHS}  lr={LR}  weight_decay={WEIGHT_DECAY}")

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

model = build_model(MODEL_CUSTOM).to(device)
optimizer = Adam(get_param_groups(model, MODEL_CUSTOM, lr_head=LR, weight_decay=WEIGHT_DECAY))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history_custom = fit(
    model, loaders,
    model_name=MODEL_CUSTOM,
    epochs=EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
)


## 10. Train: resnet18 (fine-tune ImageNet)

`torchvision.models.resnet18` с весами ImageNet. Заменён только последний `Linear` на 6 классов. Param groups: head — `LR=1e-3`, backbone — `LR=1e-4` (чтобы не сломать ImageNet-фичи). 12 эпох, CosineAnnealingLR.


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import MODEL_RESNET18
from data import get_dataloaders, load_index, make_splits
from engine import fit
from env_config import get_device
from models import build_model, get_param_groups

EPOCHS = 12
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4   # backbone предобучен на ImageNet — низкий LR, чтобы не ломать фичи
WEIGHT_DECAY = 1e-4
print(f"[{MODEL_RESNET18}] epochs={EPOCHS}  lr_head={LR_HEAD}  lr_backbone={LR_BACKBONE}  weight_decay={WEIGHT_DECAY}")

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

model = build_model(MODEL_RESNET18).to(device)
optimizer = Adam(get_param_groups(
    model, MODEL_RESNET18,
    lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history_resnet18 = fit(
    model, loaders,
    model_name=MODEL_RESNET18,
    epochs=EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
)


## 11. Train: mobilenet_v3_small (fine-tune ImageNet)

Лёгкая модель из другого семейства (depthwise-separable conv, h-swish, SE) — хорошо для сравнения «разные архитектуры». Заменён только последний `Linear` в `classifier`. Та же стратегия param groups, 12 эпох.


In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import MODEL_MOBILENETV3
from data import get_dataloaders, load_index, make_splits
from engine import fit
from env_config import get_device
from models import build_model, get_param_groups

EPOCHS = 12
LR_HEAD = 1e-3
LR_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4
print(f"[{MODEL_MOBILENETV3}] epochs={EPOCHS}  lr_head={LR_HEAD}  lr_backbone={LR_BACKBONE}  weight_decay={WEIGHT_DECAY}")

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

model = build_model(MODEL_MOBILENETV3).to(device)
optimizer = Adam(get_param_groups(
    model, MODEL_MOBILENETV3,
    lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
))
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history_mobilenet = fit(
    model, loaders,
    model_name=MODEL_MOBILENETV3,
    epochs=EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
)


## 12. Инференс и оценка на test

Грузим обученные чекпоинты с диска, считаем метрики на test, рисуем confusion matrices и показываем предсказания на случайных тестовых картинках.


In [ ]:
from config import ALL_MODELS, TARGET_COLORS
from data import get_dataloaders, load_index, make_splits
from engine import evaluate, load_checkpoint
from env_config import get_device
from models import build_model

device = get_device()
splits = make_splits(load_index())
loaders = get_dataloaders(splits)

results: dict[str, dict] = {}
for name in ALL_MODELS:
    model = build_model(name).to(device)
    meta = load_checkpoint(model, name, device=device)
    res = evaluate(model, loaders["test"], device=device, class_names=TARGET_COLORS)
    results[name] = res
    print(f"[{name}] acc={res['accuracy']:.4f}  f1_macro={res['f1_macro']:.4f}  (epochs trained: {meta['epoch']})")
    del model


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from config import FIGURE_DIR, TARGET_COLORS

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm = res["confusion_matrix"].astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{name}\nF1_macro = {res['f1_macro']:.3f}")
    ax.set_xticks(range(len(TARGET_COLORS)))
    ax.set_yticks(range(len(TARGET_COLORS)))
    ax.set_xticklabels(TARGET_COLORS, rotation=45, ha="right")
    ax.set_yticklabels(TARGET_COLORS)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{int(cm[i, j])}", ha="center", va="center", color=color, fontsize=8)

plt.tight_layout()
out = FIGURE_DIR / "confusion_matrices.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 {out}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from config import FIGURE_DIR, IMAGENET_MEAN, IMAGENET_STD, MODEL_RESNET18, TARGET_COLORS
from data import build_transforms, load_index, make_splits
from engine import load_checkpoint
from env_config import get_device
from models import build_model

device = get_device()
splits = make_splits(load_index())

# Берём pretrained resnet18 для демонстрации (обычно лучшие предсказания).
model = build_model(MODEL_RESNET18).to(device)
load_checkpoint(model, MODEL_RESNET18, device=device)
model.eval()

_, eval_tf = build_transforms()

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(splits.test), size=12, replace=False)

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, idx in zip(axes.flat, sample_idx):
    row = splits.test.iloc[int(idx)]
    img = Image.open(row["path"]).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu()
    pred_idx = int(probs.argmax())
    pred_color = TARGET_COLORS[pred_idx]
    true_color = TARGET_COLORS[int(row["label"])]
    ok = pred_idx == int(row["label"])

    show = (x.cpu()[0] * std + mean).clip(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    ax.axis("off")
    title_color = "green" if ok else "red"
    ax.set_title(f"GT: {true_color}\npred: {pred_color}  ({probs[pred_idx]:.2f})",
                 color=title_color, fontsize=10)

fig.suptitle(f"Inference: {MODEL_RESNET18}", fontsize=14)
plt.tight_layout()
out = FIGURE_DIR / "inference_examples.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 {out}")


## 13. Сравнение моделей

Сводная таблица + кривые `val_f1_macro` по эпохам + per-class F1.


In [ ]:
import pandas as pd

from models import build_model

rows = []
for name, res in results.items():
    n_params = sum(p.numel() for p in build_model(name).parameters())
    rows.append({
        "model": name,
        "params (M)": round(n_params / 1e6, 2),
        "test accuracy": round(res["accuracy"], 4),
        "test F1_macro": round(res["f1_macro"], 4),
        **{f"F1[{c}]": round(res["f1_per_class"][c], 3) for c in TARGET_COLORS},
    })

summary = pd.DataFrame(rows).sort_values("test F1_macro", ascending=False).reset_index(drop=True)
summary


In [ ]:
import json
import matplotlib.pyplot as plt

from config import LOG_DIR, ALL_MODELS, FIGURE_DIR

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for name in ALL_MODELS:
    log_path = LOG_DIR / f"{name}_history.json"
    if not log_path.exists():
        print(f"⚠️  нет {log_path}")
        continue
    with log_path.open() as f:
        hist = json.load(f)["history"]
    epochs = range(1, len(hist["val_f1_macro"]) + 1)
    ax1.plot(epochs, hist["val_loss"], label=f"{name} val")
    ax1.plot(epochs, hist["train_loss"], label=f"{name} train", linestyle="--", alpha=0.6)
    ax2.plot(epochs, hist["val_f1_macro"], label=name, linewidth=2)

ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("train/val loss"); ax1.legend(fontsize=8)
ax2.set_xlabel("epoch"); ax2.set_ylabel("val F1_macro"); ax2.set_title("validation F1_macro")
ax2.axhline(0.8, color="red", linestyle=":", label="target = 0.8")
ax2.legend()

plt.tight_layout()
out = FIGURE_DIR / "training_curves.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 {out}")


## 14. Вывод

- **Цель** — `F1_macro > 0.8` на test для задачи классификации цвета авто (DVM, top-6 цветов).
- Сравниваются три модели: `custom_resnet18` (с нуля), `resnet18` и `mobilenet_v3_small` (fine-tune с ImageNet, param groups: head LR=1e-3, backbone LR=1e-4). Все три — на одинаковом budget'е в 12 эпох.
- Pretrained-модели ожидаемо обучаются быстрее и обычно дают более высокий F1: ImageNet-фичи (формы, текстуры) переносятся хорошо, остаётся «доучить» проекцию на цвет.
- Custom from-scratch на 12 эпохах вряд ли догонит pretrained — у него нет предобученной свёрточной иерархии. Это и есть смысл сравнения: «сколько даёт сама архитектура vs сколько даёт ImageNet pretraining».
- Confusion matrix полезно посмотреть для понимания ошибок: типичные путаницы — `silver ↔ grey ↔ white` (близкие цвета на фото с разной экспозицией).

Финальные числа — в таблице раздела 13 и в `outputs/figures/`.
